# Тема 1. Радіоелектронна обстановка на цифровій карті

Ви відкрили Jupyter Notebook: файл, у якому пояснення чергуються з короткими програмами мовою Python. Програми цього файлу малюють карту навчального району з теми 1 і межу, до якої пост П1 «чує» передавачі. Змінюючи висоту антени поста, ви побачите, як зростає ця межа.

**Як працювати.** Jupyter Notebook складається з комірок: одні комірки містять текст, як ця, а інші програму. Клацніть першу комірку з програмою (вона розташована одразу під цим текстом) і натисніть Shift+Enter: комірка виконається, а під нею з’явиться результат. Так само виконуйте решту комірок по черзі, згори донизу.

In [ ]:
# Дані навчального району о 09:00. Координати в кілометрах, як на карті з підручника.
POST = {"name": "П1", "x": 6.0, "y": 16.0}
SOURCES = [
    {"name": "Д1", "what": "радіостанція командного пункту", "x": 25.0, "y": 18.0},
    {"name": "Д2", "what": "радіостанція підрозділу 1", "x": 19.0, "y": 24.0},
    {"name": "Д3", "what": "радіостанція підрозділу 2", "x": 30.0, "y": 6.0},
    {"name": "Д4", "what": "супутниковий термінал", "x": 25.2, "y": 18.1},
    {"name": "Д5", "what": "пульт керування безпілотником", "x": 15.0, "y": 16.5},
    {"name": "Д6", "what": "відео з безпілотника", "x": 10.0, "y": 19.0},
    {"name": "Д7", "what": "радар 1 (огляд повітряного простору)", "x": 44.0, "y": 22.0},
    {"name": "Д8", "what": "радар 2 (виявлення цілей)", "x": 22.0, "y": 10.0},
]
LINE = [(17.0, 32.0), (17.5, 26.0), (14.0, 19.0), (14.0, 14.0), (17.0, 8.0), (18.0, 0.0)]      # умовна лінія між сторонами
MAP_WIDTH, MAP_HEIGHT = 48.0, 32.0      # розміри району, км
LOW_ANTENNA = 2.0      # висота антени передавача, м

print("Дані району записано: пост", POST["name"], "і", len(SOURCES), "джерел.")

## Як далеко «чує» приймач

Радіохвилі цих передавачів поширюються майже прямолінійно, тому далекий передавач опиняється за опуклістю Землі. З висоти h пряма видимість сягає відстані

d = √(2 · k · R · h),

де R — це радіус Землі, а число k = 4/3 враховує, що повітря трохи вигинає радіохвилі в бік Землі. Висоту й радіус беруть у метрах, тоді й відстань виходить у метрах. Якщо над землею підняті обидві антени, і приймача, і передавача, цю відстань обчислюють для кожної антени окремо, а два результати додають. Сума двох відстаней є межею, до якої приймач «чує» передавач.

In [ ]:
import math

R_EARTH = 6371000.0      # радіус Землі, м
K = 4 / 3                # поправка на вигин радіохвиль у повітрі


def horizon_km(h):
    """Відстань до горизонту з висоти h метрів, у кілометрах."""
    return math.sqrt(2 * K * R_EARTH * h) / 1000


def reach_km(h_post, h_source=LOW_ANTENNA):
    """До якої відстані приймач з антеною на висоті h_post «чує» передавач з антеною на висоті h_source."""
    return horizon_km(h_post) + horizon_km(h_source)


def uk(number):
    """Число з одним знаком після коми."""
    return f"{number:.1f}".replace(".", ",")


print("Приймач з антеною на висоті 15 м «чує» такий передавач до", uk(reach_km(15)), "км")

## Карта

У наступній комірці є рядок `MAST = 15`. Число 15 у цьому рядку задає висоту антени поста П1 в метрах. Замініть 15 іншим числом, виконайте комірку ще раз і простежте, як змінилися межа на карті та список під нею.

Список показує лише відстань до кожного джерела і те, чи лежить воно в межах. Чи приймає пост сигнал джерела, залежить ще й від того, чи джерело випромінює і куди спрямований його промінь.

Джерела Д1 і Д4 стоять поруч, приблизно за 220 м одне від одного, тому на карті підручника в них спільна позначка. На карті, яку малює програма, їхні позначки накладаються одна на одну.

In [ ]:
import matplotlib.pyplot as plt

MAST = 15      # висота антени поста П1, м: змінюйте це число

reach = reach_km(MAST)
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot([x for x, y in LINE], [y for x, y in LINE], "--", color="#1f2a30", linewidth=1.5, label="умовна лінія")
turn = [math.radians(degree) for degree in range(361)]
ax.plot([POST["x"] + reach * math.cos(t) for t in turn], [POST["y"] + reach * math.sin(t) for t in turn],
        "--", color="#00929A", linewidth=1.5, label="межа, до якої «чує» пост")
ax.plot(POST["x"], POST["y"], "s", color="#00929A", markersize=9, label="пост П1")
ax.annotate(POST["name"], (POST["x"], POST["y"]), textcoords="offset points", xytext=(8, 6), fontweight="bold")
ax.plot([s["x"] for s in SOURCES], [s["y"] for s in SOURCES], "o", color="#C8553D", markersize=8, label="джерела")
BELOW = ["Д4"]      # ці джерела стоять упритул до інших, тому їхні підписи ставимо під позначкою
for s in SOURCES:
    ax.annotate(s["name"], (s["x"], s["y"]), textcoords="offset points",
                xytext=(8, -14) if s["name"] in BELOW else (8, 6))
ax.legend(loc="upper right", frameon=False)
ax.set_xlim(0, MAP_WIDTH)
ax.set_ylim(0, MAP_HEIGHT)
ax.set_aspect("equal")
ax.set_xticks(range(0, int(MAP_WIDTH) + 1, 4))
ax.set_yticks(range(0, int(MAP_HEIGHT) + 1, 4))
ax.grid(color="#D3DADE", linewidth=0.6)
ax.set_xlabel("x, км")
ax.set_ylabel("y, км")
ax.set_title(f"Антена поста П1 на висоті {MAST} м: межа {uk(reach)} км")
plt.show()

for s in SOURCES:
    d = math.hypot(s["x"] - POST["x"], s["y"] - POST["y"])
    print(f'{s["name"]} ({s["what"]}): {uk(d)} км від поста,', "у межах" if d <= reach else "за межею")

## Завдання

1. За якої найменшої висоти антени пост П1 прийматиме Д3? Знайдіть цю висоту, змінюючи число в рядку `MAST`.
2. Наступна комірка будує графік залежності межі від висоти антени. Знайдіть на графіку свою відповідь.
3. Антену підняли вдвічі вище, з 15 до 30 м. Чи зросла межа вдвічі? Поясніть за формулою.

In [ ]:
heights = range(2, 61)
target = next(s for s in SOURCES if s["name"] == "Д3")
to_target = math.hypot(target["x"] - POST["x"], target["y"] - POST["y"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(heights, [reach_km(h) for h in heights], color="#00929A", linewidth=2)
ax.axhline(to_target, color="#C8553D", linewidth=1.5)
ax.annotate("відстань від поста до Д3", (2.5, to_target), textcoords="offset points", xytext=(0, 5))
ax.set_xlabel("Висота антени поста П1, м")
ax.set_ylabel("Межа, до якої «чує» пост, км")
ax.set_title("Що вище антена, то далі «чує» пост")
ax.grid(color="#D3DADE", linewidth=0.6)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()